Лабораторная работа: Классификация жанров музыки через спектрограммы

Цель работы: Научиться классифицировать музыкальные жанры, преобразовав аудиосигналы в изображения (мел-спектрограммы) и применив сверточную нейросеть (CNN), которая традиционно используется для распознавания картинок.

Среда выполнения: Google Colab.

Для начала нам понадобятся библиотеки для работы с массивами, нейросетями, графиками и со звуком. Библиотека librosa — это мощный стандарт для анализа аудио в Python.

In [8]:
! pip install tensorflow 

ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)
ERROR: No matching distribution found for tensorflow


In [7]:
import os
import kagglehub
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

print("Библиотеки успешно импортированы!")

ModuleNotFoundError: No module named 'tensorflow'

Шаг 2. Загрузка датасета и выбор жанров

Мы будем использовать популярный набор данных GTZAN, который содержит по 100 треков различных музыкальных жанров (каждый трек длится 30 секунд). В датасете доступны следующие 10 жанров: blues, classical, country, disco, hiphop, jazz, metal, pop, reggae, rock.

In [ ]:
# 1. Скачиваем датасет с Kaggle
print("Загрузка датасета GTZAN с Kaggle...")
dataset_path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")
print("Датасет скачан в:", dataset_path)

# 2. Формируем путь к папке с оригинальными аудиофайлами жанров
base_path = os.path.join(dataset_path, 'Data', 'genres_original')

# Проверим, существует ли папка
if not os.path.isdir(base_path):
    # На случай, если структура немного отличается
    print("Не удалось найти папку 'Data/genres_original'. Проверяем содержимое...")
    print("Содержимое корневой папки:", os.listdir(dataset_path))
    raise FileNotFoundError("Папка с жанрами не найдена. Уточните структуру и измените путь.")

print("Путь к папкам жанров:", base_path)

# 3. Выбираем 3 жанра
genres = ['blues', 'classical', 'metal']

# Проверяем, что все выбранные жанры присутствуют
existing_genres = [g for g in genres if os.path.isdir(os.path.join(base_path, g))]
if len(existing_genres) != len(genres):
    missing = set(genres) - set(existing_genres)
    print(f"Внимание: не найдены папки для жанров {missing}")
    print("Доступные жанры:", [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))])
else:
    print("Все выбранные жанры доступны.")

print(f"Мы будем распознавать жанры: {genres}")

Шаг 3. Как выглядит музыка? Строим спектрограмму

Песня — это сложный набор звуковых волн. Чтобы нейросеть могла её понять, мы переведем звук в спектрограмму. Спектрограмма показывает, как меняются частоты (инструменты, вокал, басы) с течением времени.

Поскольку 30 секунд — это слишком большая "картинка", мы возьмем только первые 3 секунды трека.

In [ ]:
# Возьмем один блюзовый аудиофайл для примера
sample_dir = os.path.join(base_path, 'blues')
sample_audio_path = os.path.join(sample_dir, os.listdir(sample_dir)[0])

# Загружаем аудио (sr=22050 - стандартная частота, duration=3.0 - берем только 3 секунды!)
y, sr = librosa.load(sample_audio_path, sr=22050, duration=3.0)

# Генерируем Мел-спектрограмму (адаптировано под человеческое восприятие слуха)
mel_spectrogram = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)

# Переводим амплитуду в децибелы (логарифмическая шкала)
mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)

# Рисуем нашу "картинку"
plt.figure(figsize=(10, 4))
librosa.display.specshow(mel_spectrogram_db, x_axis='time', y_axis='mel', sr=sr, fmax=8000)
plt.colorbar(format='%+2.0f dB')
plt.title('Мел-спектрограмма жанра "Blues" (фрагмент 3 сек)')
plt.tight_layout()
plt.show()

Шаг 4. Подготовка данных для нейросети

Теперь мы проделаем эту операцию для всех файлов в наших папках и соберем их в массивы X (спектрограммы) и Y (метки жанров).

In [ ]:
def audio_to_spectrogram(file_path):
    # Загружаем 3 секунды аудио
    y, sr = librosa.load(file_path, sr=22050, duration=3.0)

    # Делаем длину всех записей ровно 3 секунды (22050 * 3 сэмплов)
    target_length = 22050 * 3
    if len(y) < target_length:
        y = np.pad(y, (0, target_length - len(y)))
    else:
        y = y[:target_length]

    # Вычисляем спектрограмму
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64, hop_length=512)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

    # Добавляем "канал" (как в RGB, но у нас черно-белое, поэтому 1 канал)
    return mel_spec_db[..., np.newaxis]

X = []
y_labels = []

print("Начинаем обработку файлов. Это займет несколько минут...")

for i, genre in enumerate(genres):
    folder_path = os.path.join(base_path, genre)
    files = os.listdir(folder_path)
    # В каждой папке по 100 треков
    for file_name in files:
        file_path = os.path.join(folder_path, file_name)
        if file_path.endswith('.au') or file_path.endswith('.wav'):
            spec = audio_to_spectrogram(file_path)
            X.append(spec)
            y_labels.append(i) # 0, 1, 2 или 3

X = np.array(X)
y_labels = np.array(y_labels)

# Нормализуем данные для лучшего обучения нейросети (приводим значения от 0 до 1)
X = (X - np.min(X)) / (np.max(X) - np.min(X))

# Разбиваем на обучающую и тестовую выборки (80% на обучение, 20% на тест)
X_train, X_test, y_train, y_test = train_test_split(X, y_labels, test_size=0.2, random_state=42)

print(f"Данные готовы! Форма X_train: {X_train.shape} (кол-во треков, высота, ширина, каналы)")

Шаг 5. Создание сверточной нейросети (CNN)

Создадим модель компьютерного зрения, которая будет искать характерные паттерны жанров на наших спектрограммах (например, частые вертикальные полосы — это биты).

In [ ]:
input_shape = X_train.shape[1:] # Узнаем размер одной спектрограммы

model = models.Sequential([
    # Сверточные слои ищут паттерны (ритм, басы, переходы) на спектрограмме
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Вытягиваем двумерную картинку в одномерный вектор
    layers.Flatten(),

    # Полносвязные слои для принятия финального решения
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), # Защита от переобучения (чтобы сеть не "зазубривала" треки)
    layers.Dense(len(genres), activation='softmax') # На выходе 4 класса
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Шаг 6. Обучение и тестирование модели

Запускаем обучение нашей нейросети

In [ ]:
# Обучаем модель (epochs - количество проходов по всем данным)
history = model.fit(X_train, y_train,
                    epochs=20,
                    validation_data=(X_test, y_test),
                    batch_size=32)

# Оценка точности
test_loss, test_acc = model.evaluate(X_test,  y_test, verbose=2)
print(f'\nТочность на тестовых данных: {test_acc * 100:.2f}%')

In [ ]:
plt.plot(history.history['accuracy'], label='Точность на обучении')
plt.plot(history.history['val_accuracy'], label='Точность на валидации')
plt.xlabel('Эпоха')
plt.ylabel('Точность')
plt.legend(loc='lower right')
plt.title('График обучения нейросети')
plt.show()

Шаг 7. Проверка модели на собственной песне

загрузим свой аудиофайл в Google Colab и посмотрим, сможет ли модель правильно определить его жанр.

In [ ]:
from google.colab import files

print("Загрузите ваш аудиофайл (mp3 или wav):")
uploaded = files.upload()

for file_name in uploaded.keys():
    print(f'\nАнализируем файл: "{file_name}"...')

    # Получаем спектрограмму загруженной песни
    my_song_spec = audio_to_spectrogram(file_name)

    # Добавляем измерение батча (модель ожидает массив картинок, а не одну)
    my_song_batch = np.expand_dims(my_song_spec, axis=0)

    # Нормализуем конкретно эту картинку (как делали на шаге 4)
    # Добавляем 1e-10, чтобы избежать деления на ноль в случае тишины
    my_song_batch = (my_song_batch - np.min(my_song_batch)) / (np.max(my_song_batch) - np.min(my_song_batch) + 1e-10)

    # Делаем предсказание
    predictions = model.predict(my_song_batch)

    # Находим индекс максимальной вероятности
    predicted_index = np.argmax(predictions[0])
    predicted_genre = genres[predicted_index]
    confidence = predictions[0][predicted_index] * 100

    print(f"🎵 Нейросеть считает, что это: {predicted_genre.upper()} (Уверенность: {confidence:.2f}%)")

    # Выводим вероятности для всех жанров
    print("\nПодробный расклад вероятностей:")
    for i, genre in enumerate(genres):
        print(f"{genre}: {predictions[0][i]*100:.2f}%")

Задания для самостоятельной работы

1. Добавление всех жанров датасета: В датасете GTZAN целых 10 жанров (указаны в Шаге 2). Добавьте в массив genres стили jazz, hiphop, rock и reggae. Обучите модель заново. Насколько упала точность при увеличении числа классов?

2. Анализ ошибок (Матрица ошибок): Модель почти никогда не работает со 100% точностью. С помощью библиотеки sklearn постройте матрицу ошибок (confusion_matrix), чтобы визуально увидеть, какие жанры ИИ путает чаще всего.

3. Середина трека вместо вступления: В функции audio_to_spectrogram мы берем первые 3 секунды трека, но часто в начале песни звучит долгое тихое вступление. Изучите документацию librosa.load и используйте параметр offset=15.0, чтобы вырезать фрагмент с 15-й секунды. Улучшит ли это распознавание?

4. Архитектурные эксперименты: Попробуйте изменить структуру нейросети на Шаге 5. Что произойдет, если убрать один из слоев Conv2D? А если заменить функцию активации relu на selu или увеличить количество эпох (epochs) до 50?

5. Другие аудио-признаки: Помимо мел-спектрограммы, в анализе звука часто используют MFCC (Mel-frequency cepstral coefficients). Попробуйте заменить librosa.feature.melspectrogram на librosa.feature.mfcc в функции подготовки данных. Сравните результаты двух подходов.